## This analysis seeks to answer the hypothesis: How much will we sell in the coming months?

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName("Sales_Prediction") \
        .config(
        "spark.jars.packages",
        "org.postgresql:postgresql:42.7.3") \
        .getOrCreate()

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
database = os.getenv("DB_NAME")

try:
    query="""
    (select
      date_trunc('month', order_purchase_timestamp) as Month,
      sum(price) as total_revenue
    from
     analytics.fact_orders
     group by Month  
     order by Month) as Revenue_query
     """
     
    sales_df = spark.read.format("jdbc") \
        .option("url", f"jdbc:postgresql://{host}:{port}/{database}") \
        .option("dbtable", query) \
        .option("user", user) \
        .option("password", password) \
        .option("driver", "org.postgresql.Driver") \
        .load()
    
    sales_df.show(5)
    
except Exception as e:
    print(f"The error is detailed: {e}")



    



+-------------------+--------------------+
|              month|       total_revenue|
+-------------------+--------------------+
|2016-09-01 00:00:00|267.3600000000000...|
|2016-10-01 00:00:00|49507.66000000000...|
|2016-12-01 00:00:00|10.90000000000000...|
|2017-01-01 00:00:00|120312.8700000000...|
|2017-02-01 00:00:00|247303.0200000000...|
+-------------------+--------------------+
only showing top 5 rows


In [5]:
sales_df.printSchema()

root
 |-- month: timestamp (nullable = true)
 |-- total_revenue: decimal(38,18) (nullable = true)



In [6]:
sales_df.summary().show()

+-------+--------------------+
|summary|       total_revenue|
+-------+--------------------+
|  count|                  24|
|   mean|566318.4875000000...|
| stddev|   356670.4854494177|
|    min|10.90000000000000...|
|    25%|           247303.02|
|    50%|           573971.68|
|    75%|           865124.31|
|    max|1010271.370000000...|
+-------+--------------------+

